# QCC baseline + verification harness (team notebook)

1. Rebuilds the best **public** portfolio (from the public dataset `heliosli/qcc-circuits-live`,
   Apache-2.0 components credited in that dataset's manifest and in heliosli's notebook *The dataset is public now*).
2. Re-verifies every task with the **unmodified official verifiers** under our *own* random salt.
3. Provides the harness functions (`score_task`, `profile_circuit`) used for all of our experiments.
4. Writes `/kaggle/working/submission.zip` and `results.json`.

Credits: circuit lineage as documented in the release manifest and in
https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy.

In [ ]:
import glob, hashlib, json, lzma, math, os, shutil, sys, time, zipfile, pathlib, secrets, collections

def find_comp():
    for p in [os.environ.get('QCC_COMP_DIR', ''),
              '/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation',
              '/kaggle/input/google-verifiable-quantum-circuit-compilation']:
        if p and os.path.isdir(os.path.join(p, 'verifiers')):
            return p
    hits = glob.glob('/kaggle/input/**/verifiers/invsqrt', recursive=True)
    if hits:
        return os.path.dirname(os.path.dirname(hits[0]))
    raise SystemExit('competition data not found')

def find_release():
    hits = glob.glob('/kaggle/input/**/QCC_RELEASE_MANIFEST.json', recursive=True)
    if not hits:
        raise SystemExit('release dataset heliosli/qcc-circuits-live not attached')
    return os.path.dirname(hits[0])

COMP, DS = find_comp(), find_release()
WORK = pathlib.Path('/kaggle/working')
SUB = WORK / 'submission'
sys.path[:0] = [os.path.join(COMP, 'kickmix'), os.path.join(COMP, 'verifiers')]
import kickmix as km
from invsqrt import verify_submission_invsqrt as vinv
from dqi import verify_submission_dqi as vdqi
from dfthcsos import verify as vdft
MANIFEST = json.load(open(os.path.join(DS, 'QCC_RELEASE_MANIFEST.json')))
REF = MANIFEST['reference_costs']
OUR_SALT = os.environ.get('QCC_SALT') or ('team-' + secrets.token_hex(8))
print('comp:', COMP, '| release:', DS, '| portfolio:', MANIFEST.get('portfolio'), '| our salt:', OUR_SALT)
print('reference costs:', REF)

## Materialise the public portfolio (SHA-256 checked)

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()

if SUB.exists():
    shutil.rmtree(SUB)
for item in MANIFEST['files']:
    src, dst = os.path.join(DS, item['stored_as']), SUB / item['file']
    dst.parent.mkdir(parents=True, exist_ok=True)
    if item['stored_as'].endswith('.xz'):
        with lzma.open(src) as fi, open(dst, 'wb') as fo:
            shutil.copyfileobj(fi, fo, 1 << 22)
    else:
        shutil.copyfile(src, dst)
    assert sha256(dst) == item['sha256'], item['file']
print(len(MANIFEST['files']), 'files OK')

## Harness: score any candidate with the unmodified verifiers

In [ ]:
def _load(path):
    path = pathlib.Path(path)
    return km.Circuit.from_file(str(path), format=path.suffix.removeprefix('.'))

def score_task(task, path, salt=None):
    """Return (ratio, result_dict). `path`: .kmb/.kmx file (invsqrt, dqi) or dfthcsos folder."""
    salt = OUR_SALT if salt is None else salt
    t0 = time.time()
    if task == 'invsqrt':
        r = vinv.verify(_load(path), vinv.ProblemSpecInvSqrt.for_n_p(10), salt=salt)
    elif task == 'dqi':
        r = vdqi.verify(_load(path), vdqi.ProblemSpecDQI.benchmark_instance(), salt=salt)
    elif task == 'dfthcsos':
        r = vdft.verify(str(path), salt=salt)
    else:
        raise ValueError(task)
    assert r['status'] == 'PASS', (task, r)
    ratio = REF[task] / float(r['score'])
    print(f"{task:9s} PASS  cost {float(r['score']):.6g}  ratio {ratio:.4f}  ({time.time()-t0:.0f}s)")
    return ratio, r

def geo_mean(ratios):
    return math.prod(ratios) ** (1 / len(ratios))

## Profiler: instruction mix of a circuit (text form)

In [ ]:
def profile_circuit(path, top=25):
    """Converts to .kmx text and counts instruction mnemonics; quick look at where gates go."""
    c = _load(path)
    tmp = WORK / (pathlib.Path(path).stem + '_profile.kmx')
    c.to_file(str(tmp), format='kmx')
    counts, lines = collections.Counter(), 0
    with open(tmp) as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith('#'):
                continue
            lines += 1
            counts[s.split()[0].split('(')[0]] += 1
    tmp.unlink()
    print(f'{path}: {lines} instructions')
    for k, v in counts.most_common(top):
        print(f'  {k:24s} {v}')
    return counts

profile_circuit(SUB / 'invsqrt.kmb')

## Verify the portfolio under our own salt

In [ ]:
results, ratios = {}, []
for task, p in [('invsqrt', SUB / 'invsqrt.kmb'), ('dqi', SUB / 'dqi.kmb'), ('dfthcsos', SUB / 'dfthcsos')]:
    ratio, r = score_task(task, p)
    ratios.append(ratio)
    results[task] = {'ratio': ratio, **{k: (v if isinstance(v, (int, float, str, bool)) else str(v)) for k, v in r.items()}}
gm = geo_mean(ratios)
print(f'GEOMETRIC MEAN (our salt): {gm:.5f}')
json.dump({'salt': OUR_SALT, 'portfolio': MANIFEST.get('portfolio'), 'geo_mean': gm, 'tasks': results},
          open(WORK / 'results.json', 'w'), indent=1)

## Write submission.zip

In [ ]:
zip_path = WORK / 'submission.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for item in MANIFEST['files']:
        z.write(SUB / item['file'], item['file'])
shutil.rmtree(SUB)
print('wrote', zip_path, round(zip_path.stat().st_size / 2**20, 1), 'MiB')